<a href="https://colab.research.google.com/github/sunilbarigala25/tcgenerator-colab-gradio/blob/main/TCGen_MultiModel_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gradio as gr
from openai import OpenAI
import json
import re
import os
import pandas as pd
from datetime import datetime
import google.generativeai as genai
import time

# Import Bytez SDK (needs to be installed: pip install bytez)
try:
    from bytez import Bytez
    BYTEZ_AVAILABLE = True
except ImportError:
    BYTEZ_AVAILABLE = False
    print("⚠️ WARNING: Bytez SDK not installed. Install with: pip install bytez")


# -------------------------
# BUILT-IN API KEYS FOR FREE MODELS
# -------------------------
BYTEZ_FREE_API_KEY = "96081f1caf99eeba4de3a27278a2eff4"

# Model configuration with API requirements
MODEL_CONFIG = {
    "Bytez GPT-4o-mini (FREE)": {
        "requires_api_key": False,
        "model_name": "openai/gpt-4o-mini",
        "api_key": BYTEZ_FREE_API_KEY,
        "api_type": "bytez"
    },
    "Bytez Gemma-3-1b (FREE)": {
        "requires_api_key": False,
        "model_name": "google/gemma-3-1b-it",
        "api_key": BYTEZ_FREE_API_KEY,
        "api_type": "bytez"
    },
    "ChatGPT": {
        "requires_api_key": True,
        "model_name": "gpt-4o",
        "api_type": "openai"
    },
    "Gemini": {
        "requires_api_key": True,
        "model_name": "gemini-pro",
        "api_type": "gemini"
    },
    "Copilot": {
        "requires_api_key": True,
        "model_name": "gpt-4",
        "api_type": "copilot"
    },
    "OpenRouter": {
        "requires_api_key": True,
        "model_name": "openai/gpt-4o",
        "api_type": "openrouter"
    }
}


# -------------------------
# Console Logger with Copy Function
# -------------------------
class ConsoleLogger:
    def __init__(self):
        self.logs = []
        self.plain_logs = []

    def success(self, message):
        timestamp = datetime.now().strftime('%H:%M:%S')
        msg = f'<span style="color: #16a34a;">[{timestamp}] ✅ SUCCESS: {message}</span>'
        self.logs.append(msg)
        self.plain_logs.append(f"[{timestamp}] ✅ SUCCESS: {message}")
        print(f"✅ SUCCESS: {message}")

    def error(self, message):
        timestamp = datetime.now().strftime('%H:%M:%S')
        msg = f'<span style="color: #dc2626;">[{timestamp}] ❌ ERROR: {message}</span>'
        self.logs.append(msg)
        self.plain_logs.append(f"[{timestamp}] ❌ ERROR: {message}")
        print(f"❌ ERROR: {message}")

    def warning(self, message):
        timestamp = datetime.now().strftime('%H:%M:%S')
        msg = f'<span style="color: #ea580c;">[{timestamp}] ⚠️ WARNING: {message}</span>'
        self.logs.append(msg)
        self.plain_logs.append(f"[{timestamp}] ⚠️ WARNING: {message}")
        print(f"⚠️ WARNING: {message}")

    def info(self, message):
        timestamp = datetime.now().strftime('%H:%M:%S')
        msg = f'<span style="color: #2563eb;">[{timestamp}] ℹ️ INFO: {message}</span>'
        self.logs.append(msg)
        self.plain_logs.append(f"[{timestamp}] ℹ️ INFO: {message}")
        print(f"ℹ️ INFO: {message}")

    def debug(self, message):
        timestamp = datetime.now().strftime('%H:%M:%S')
        msg = f'<span style="color: #6b7280;">[{timestamp}] 🔍 DEBUG: {message}</span>'
        self.logs.append(msg)
        self.plain_logs.append(f"[{timestamp}] 🔍 DEBUG: {message}")
        print(f"🔍 DEBUG: {message}")

    def progress(self, message):
        timestamp = datetime.now().strftime('%H:%M:%S')
        msg = f'<span style="color: #06b6d4;">[{timestamp}] 🔄 PROGRESS: {message}</span>'
        self.logs.append(msg)
        self.plain_logs.append(f"[{timestamp}] 🔄 PROGRESS: {message}")
        print(f"🔄 PROGRESS: {message}")

    def get_html(self):
        logs_html = "<br>".join(self.logs)
        plain_text = "\n".join(self.plain_logs)
        plain_text_escaped = plain_text.replace('\\', '\\\\').replace("'", "\\'").replace('\n', '\\n').replace('"', '\\"')

        return f"""
        <div style="position: relative;">
            <button id="copy-logs-btn" onclick="copyConsoleLogs()" style="
                position: absolute;
                top: 8px;
                right: 8px;
                background: #3b82f6;
                color: white;
                border: none;
                padding: 6px 12px;
                border-radius: 6px;
                cursor: pointer;
                font-size: 12px;
                font-weight: 600;
                z-index: 100;
            " onmouseover="this.style.background='#2563eb'" onmouseout="this.style.background='#3b82f6'">
                📋 Copy Logs
            </button>
            <div class='console-box' id='console-logs-content'>{logs_html}</div>
        </div>
        <script>
        function copyConsoleLogs() {{
            const text = `{plain_text_escaped}`;
            const decoded = text.replace(/\\\\n/g, '\\n').replace(/\\\\'/g, "'").replace(/\\\\"/g, '"');

            navigator.clipboard.writeText(decoded).then(() => {{
                const btn = document.getElementById('copy-logs-btn');
                const originalText = btn.innerHTML;
                btn.innerHTML = '✅ Copied!';
                btn.style.background = '#16a34a';
                setTimeout(() => {{
                    btn.innerHTML = originalText;
                    btn.style.background = '#3b82f6';
                }}, 2000);
            }}).catch(err => {{
                console.error('Failed to copy:', err);
                const consoleText = document.getElementById('console-logs-content').innerText;
                navigator.clipboard.writeText(consoleText).then(() => {{
                    alert('✅ Logs copied to clipboard!');
                }}).catch(() => {{
                    alert('❌ Failed to copy logs. Please select and copy manually.');
                }});
            }});
        }}
        </script>
        """

    def clear(self):
        self.logs = []
        self.plain_logs = []


# -------------------------
# IMPROVED BASE TEST CASE PROMPT
# -------------------------
DEFAULT_MASTER_PROMPT = """
You are a Senior Software Test Architect with 15+ years of experience in
designing comprehensive test strategies for large-scale applications. You
specialize in UAT, Integration, Performance, Security, and Regression testing.

Your expertise includes:
- Deep understanding of software architecture, APIs, databases, and cloud systems
- Proficiency in identifying edge cases, race conditions, and system bottlenecks
- Experience with modern frameworks (React, Node.js, microservices, GraphQL, REST APIs)
- Knowledge of CI/CD pipelines, containerization (Docker/K8s), and DevOps practices
- Expertise in data validation, security testing, and compliance requirements

CRITICAL INSTRUCTIONS:
Generate COMPREHENSIVE, PRODUCTION-GRADE UAT test cases that go far beyond basic
functional testing. Each test case should demonstrate deep technical understanding
and cover real-world production scenarios.

TEST COVERAGE REQUIREMENTS (MANDATORY):

1. FUNCTIONAL TESTING (Core Features):
   - Happy path scenarios with ALL valid input combinations
   - Business logic validation with complex workflows
   - Data integrity checks across multiple system layers
   - Transaction validation (ACID properties where applicable)
   - State management and session handling

2. NEGATIVE TESTING (Robust Error Handling):
   - Invalid inputs (null, empty, special characters, injection attempts)
   - Boundary violations (min/max limits, overflow scenarios)
   - Concurrent user actions and race conditions
   - Permission and authorization violations
   - Malformed API requests and response handling
   - Network failures and timeout scenarios

3. INTEGRATION TESTING (System Interactions):
   - API contract validation (request/response schemas)
   - Database transaction verification (CRUD operations)
   - Third-party service integration (payment gateways, auth providers)
   - Message queue/event-driven architecture validation
   - Data synchronization across microservices
   - Cache coherency and invalidation strategies

4. UI/UX TESTING (User Experience):
   - Responsive design across devices (mobile, tablet, desktop)
   - Accessibility compliance (WCAG 2.1 AA standards)
   - Loading states, spinners, and user feedback mechanisms
   - Form validation (client-side and server-side)
   - Error message clarity and actionable guidance
   - Navigation flow and breadcrumb accuracy

5. PERFORMANCE TESTING (Scalability):
   - Load testing with concurrent users (10, 100, 1000+ users)
   - Response time validation (API calls < 2s, page loads < 3s)
   - Database query optimization verification
   - CDN and static asset caching effectiveness
   - Memory leak detection and resource cleanup
   - Stress testing under peak load conditions

6. SECURITY TESTING (Vulnerability Assessment):
   - Authentication and authorization mechanisms
   - XSS (Cross-Site Scripting) prevention
   - CSRF (Cross-Site Request Forgery) protection
   - Injection prevention (SQL, NoSQL, Command)
   - Sensitive data encryption (at rest and in transit)
   - API rate limiting and DDoS protection
   - Session management and token expiration

7. DATA VALIDATION (Accuracy & Consistency):
   - Input sanitization and validation rules
   - Data format consistency (dates, currencies, phone numbers)
   - Duplicate detection and prevention
   - Data migration and transformation accuracy
   - Referential integrity in database relationships
   - Audit trail and logging completeness

8. BACKWARD COMPATIBILITY (Version Management):
   - API versioning and deprecation handling
   - Database schema migration without data loss
   - Support for older client versions
   - Feature flag and A/B testing scenarios
   - Rollback procedures and data recovery

9. EDGE CASES & BOUNDARY CONDITIONS:
   - Zero/empty/null data handling
   - Maximum character limits (text fields, file uploads)
   - Timezone and locale-specific scenarios
   - Network latency and offline mode behavior
   - Browser compatibility (Chrome, Firefox, Safari, Edge)
   - Mobile OS versions (iOS, Android)

10. COMPLIANCE & REGULATORY (Industry Standards):
    - GDPR compliance (data privacy, right to erasure)
    - PCI DSS for payment processing
    - HIPAA for healthcare data (if applicable)
    - SOC 2 Type II compliance
    - Logging and audit requirements

EACH TEST CASE MUST INCLUDE:
- Clear preconditions and environment setup
- Specific test data with realistic values
- Detailed step-by-step execution instructions
- Expected results with measurable success criteria
- Postconditions and cleanup procedures
- Risk level (Critical/High/Medium/Low)
- Test type classification (Functional/Integration/Security/Performance)

REQUIRED JSON STRUCTURE:
unique_id: Sequential integer starting from 1 (includes test case + all steps)
type: "test_manual" for test case row, "step" for step rows
step_type: "simple" for all step rows, empty for test case row
name: "JIRAID_UAT_TC001" format (only for test_manual rows)
teams_udf: Leave blank
testcasetype_udf: "Functional", "Integration", "Security", "Performance", "UI", or combination
testphase_udf: "UAT"
dmnum_udf: JIRA Story ID (e.g., "LRECOM-16429")
testjourney_udf: Leave blank
description: Concise summary (max 150 chars) of what is being tested
criticality_udf: "Critical", "High", "Medium", or "Low"
prerequisites_udf: Detailed preconditions (user roles, data setup, environment config)
testdata_udf: Specific test data values, user types, API endpoints, database states
circle_udf: Leave blank unless geo-specific testing required
step_description: Numbered steps with final step containing "Expected Result:"

**CRITICAL OUTPUT REQUIREMENT:**
You MUST output ONLY a valid JSON array. Start with [ and end with ].
NO markdown code blocks (```json), NO explanations, NO text before or after.
Just pure JSON array: [ {...}, {...}, ... ]

Generate 20-35 comprehensive test cases covering ALL the testing areas mentioned above.
Think like a senior QA architect designing for a production system handling millions of users.
Consider real-world scenarios: system failures, peak loads, security threats, data corruption.
"""

# -------------------------
# Helper utilities
# -------------------------
EXPECTED_COLS = [
    "unique_id", "type", "step_type", "name", "teams_udf",
    "testcasetype_udf", "testphase_udf", "dmnum_udf",
    "testjourney_udf", "description", "criticality_udf",
    "prerequisites_udf", "testdata_udf", "circle_udf",
    "step_description"
]


def extract_json_from_text(text, logger):
    """Enhanced JSON extraction with debug logging"""
    logger.debug(f"Attempting to extract JSON from text (length: {len(text)} chars)")
    text = text.strip()

    try:
        logger.debug("Strategy 1: Trying direct JSON parse...")
        result = json.loads(text)
        logger.success("Direct JSON parse successful!")
        return result
    except Exception as e:
        logger.warning(f"Direct JSON parse failed: {str(e)[:100]}")

    logger.debug("Strategy 2: Looking for ```json code block...")
    m = re.search(r"```json\s*(.*?)\s*```", text, re.S | re.I)
    if m:
        try:
            result = json.loads(m.group(1).strip())
            logger.success("Found JSON in ```json code block!")
            return result
        except Exception as e:
            logger.warning(f"JSON in code block is invalid: {str(e)[:100]}")

    logger.debug("Strategy 3: Looking for any code block...")
    m2 = re.search(r"```\s*(.*?)\s*```", text, re.S)
    if m2:
        try:
            result = json.loads(m2.group(1).strip())
            logger.success("Found JSON in generic code block!")
            return result
        except Exception as e:
            logger.warning(f"Generic code block JSON invalid: {str(e)[:100]}")

    logger.debug("Strategy 4: Searching for JSON-like array or object...")
    first = min([i for i in (text.find('['), text.find('{')) if i != -1], default=-1)
    if first != -1:
        for lastchar in (']', '}'):
            last = text.rfind(lastchar)
            if last > first:
                candidate = text[first:last+1]
                try:
                    result = json.loads(candidate)
                    logger.success(f"Found valid JSON between {first} and {last}!")
                    return result
                except Exception as e:
                    logger.debug(f"Candidate JSON invalid: {str(e)[:50]}")

    logger.error("All JSON extraction strategies failed!")
    logger.debug(f"First 500 chars of response: {text[:500]}")
    return None


def create_fallback_json_from_story(story, jira_id, logger):
    """
    Generate IMPROVED fallback JSON with better test cases

    🔍 FALLBACK MODE EXPLANATION:
    This function activates when the AI API fails or returns invalid JSON.
    It creates test cases by analyzing the user story text directly using Python logic.

    📋 HOW IT WORKS:

    1. STORY PARSING:
       - Splits the user story into lines
       - Extracts the title (first line)
       - Looks for "Acceptance Criteria" section
       - Parses bullet points as individual criteria

    2. TEST CASE GENERATION LOGIC:
       - Creates 25 test cases by default (or 3x acceptance criteria count)
       - Cycles through 8 test types: Functional, Integration, Negative, Performance,
         Security, UI/UX, Data Validation, Edge Cases
       - Assigns criticality levels in a pattern: Critical, High, High, Medium...

    3. TEST CASE STRUCTURE:
       For each test case:
       - Creates a test_manual row with metadata
       - Generates 7 detailed steps based on test type
       - Each step is specific to the test type (e.g., Performance steps check load,
         Security steps check authentication, etc.)

    4. PREDEFINED STEP TEMPLATES:
       - Functional: Login → Navigate → Perform action → Verify → Check UI → API validation
       - Integration: API check → Request → Response time → Database → Event queue
       - Negative: Invalid inputs → Injection attempts → Boundary tests → Auth failures
       - Performance: Load testing → Concurrent users → Response times → Resource usage
       - Security: Authentication → Authorization → Encryption → Session management
       - UI/UX: Responsive design → Accessibility → Error messages → Browser compatibility

    5. FINAL OUTPUT:
       - Each test case has 8 rows total (1 test_manual + 7 steps)
       - unique_id is sequential across all rows
       - Includes realistic test data, prerequisites, and expected results

    ⚠️ LIMITATIONS:
    - Generic test cases not tailored to specific feature
    - No AI understanding of business logic
    - Fixed step templates for each test type
    - Cannot extract context from uploaded design docs

    ✅ WHEN IT'S USEFUL:
    - API rate limits exceeded
    - Network connectivity issues
    - Invalid API keys
    - AI service downtime
    - Still provides professional-looking test template
    """
    logger.warning("Entering IMPROVED fallback mode - generating advanced template test cases")

    lines = [ln.strip() for ln in story.splitlines() if ln.strip()]
    logger.debug(f"Story has {len(lines)} non-empty lines")

    title = lines[0] if lines else "Test Case from User Story"
    if len(title) > 200:
        title = title[:200]

    acceptance_criteria = []
    in_criteria = False
    for line in lines:
        if "acceptance criteria" in line.lower() or "ac:" in line.lower():
            in_criteria = True
            continue
        if in_criteria and (line.startswith("-") or line.startswith("*")):
            criteria = line.lstrip("- *").strip()
            acceptance_criteria.append(criteria)

    logger.info(f"Extracted {len(acceptance_criteria)} acceptance criteria")

    fallback_json = []
    tc_count = max(15, len(acceptance_criteria) * 3) if acceptance_criteria else 25
    logger.info(f"Generating {tc_count} IMPROVED fallback test cases")

    test_types = ["Functional", "Integration", "Negative Testing", "Performance",
                  "Security", "UI/UX", "Data Validation", "Edge Cases"]
    criticality_levels = ["Critical", "High", "High", "Medium", "High", "Medium", "Medium", "Low"]

    for i in range(1, tc_count + 1):
        unique_id_start = (i - 1) * 8 + 1

        test_type = test_types[(i-1) % len(test_types)]
        criticality = criticality_levels[(i-1) % len(criticality_levels)]

        if acceptance_criteria and i <= len(acceptance_criteria):
            base_desc = acceptance_criteria[i-1][:80]
        else:
            base_desc = f"{title[:60]}"

        description = f"Verify {test_type}: {base_desc}"
        tc_name = f"{jira_id}_UAT_TC{str(i).zfill(3)}" if jira_id else f"UAT_TC{str(i).zfill(3)}"

        prerequisites = (
            f"1) System configured in UAT environment with latest build "
            f"2) Test user account with appropriate permissions created "
            f"3) Database populated with test data (minimum 100 records) "
            f"4) API endpoints verified and accessible "
            f"5) Logging and monitoring tools enabled"
        )

        testdata = (
            f"Test User: qa_user_{i}@testdomain.com, "
            f"Test Environment: UAT, "
            f"Browser: Chrome v120+, "
            f"Test Data Set: {test_type.replace(' ', '_').lower()}_dataset_{i}, "
            f"API Endpoint: /api/v1/feature, "
            f"Database: test_db_uat"
        )

        fallback_json.append({
            "unique_id": unique_id_start,
            "type": "test_manual",
            "step_type": "",
            "name": tc_name,
            "teams_udf": "",
            "testcasetype_udf": test_type,
            "testphase_udf": "UAT",
            "dmnum_udf": jira_id or "",
            "testjourney_udf": "",
            "description": description,
            "criticality_udf": criticality,
            "prerequisites_udf": prerequisites,
            "testdata_udf": testdata,
            "circle_udf": "",
            "step_description": ""
        })

        if "Functional" in test_type:
            steps = [
                "1. Login to the application using valid test credentials and verify successful authentication",
                "2. Navigate to the feature module as specified in user story and verify page loads within 3 seconds",
                "3. Perform the primary action as described in acceptance criteria with valid input data",
                "4. Verify system processes the action correctly and updates database with transaction ID logged",
                "5. Check UI reflects the changes immediately with appropriate success message displayed",
                "6. Validate API response returns HTTP 200 status with correct JSON schema",
                "7. Expected Result: Feature functions as per acceptance criteria, all data persists correctly, no errors in console logs, audit trail captured, user receives confirmation notification"
            ]
        elif "Integration" in test_type:
            steps = [
                "1. Verify API endpoint is accessible and returns correct status code (200/201)",
                "2. Send valid API request with required headers (Authorization, Content-Type) and body parameters",
                "3. Verify response time is under 2 seconds and response payload matches expected schema",
                "4. Check database records created/updated with correct values and foreign key relationships maintained",
                "5. Verify event published to message queue (if applicable) with correct payload structure",
                "6. Test integration with third-party services (if any) and verify callback handling",
                "7. Expected Result: Complete end-to-end data flow works correctly, all systems synchronized, transaction consistency maintained, proper error handling for service failures"
            ]
        elif "Negative" in test_type:
            steps = [
                "1. Attempt operation with missing required fields and capture error response",
                "2. Submit invalid data formats (special characters, injection strings, XSS payloads)",
                "3. Test with boundary values (negative numbers, zero, maximum integers, empty strings)",
                "4. Verify unauthorized access attempts are blocked with 401/403 error codes",
                "5. Test concurrent operations by same user to check for race conditions",
                "6. Simulate network timeout and verify graceful degradation with retry mechanism",
                "7. Expected Result: System handles all invalid inputs gracefully, returns appropriate error messages, logs security events, no system crashes or data corruption, user-friendly error guidance provided"
            ]
        elif "Performance" in test_type:
            steps = [
                "1. Set up load testing tool (JMeter/k6) with 100 concurrent virtual users",
                "2. Execute sustained load test for 5 minutes with ramp-up time of 1 minute",
                "3. Monitor API response times using APM tools and verify 95th percentile < 2 seconds",
                "4. Check database connection pool utilization stays below 80% threshold",
                "5. Verify CPU and memory usage remain stable without memory leaks",
                "6. Test cache hit ratio and verify it exceeds 70% for repeated requests",
                "7. Expected Result: System handles target load without degradation, response times within SLA, no timeout errors, resource utilization optimal, all transactions complete successfully"
            ]
        elif "Security" in test_type:
            steps = [
                "1. Verify authentication mechanism uses secure protocols (OAuth 2.0/JWT tokens)",
                "2. Test authorization by attempting to access resources without proper permissions",
                "3. Inject malicious payloads and verify sanitization works correctly",
                "4. Check sensitive data is encrypted in transit (HTTPS) and at rest (AES-256)",
                "5. Verify session management with proper timeout (15 minutes inactivity) and token expiration",
                "6. Test rate limiting to prevent brute force attacks (max 10 login attempts per minute)",
                "7. Expected Result: All security controls active, no vulnerabilities exploitable, PII data masked in logs, security headers present, failed attempts logged for audit"
            ]
        else:
            steps = [
                "1. Login with valid test account and navigate to the target feature page",
                "2. Verify all UI elements render correctly across different screen sizes (mobile, tablet, desktop)",
                "3. Test user interaction flows including form validation, error messages, and loading states",
                "4. Perform the main feature action and verify real-time UI updates without page refresh",
                "5. Check accessibility compliance (keyboard navigation, screen reader support, ARIA labels)",
                "6. Test edge cases including empty data sets, maximum character limits, special characters in input",
                "7. Expected Result: UI is responsive and intuitive, all validations work client and server side, error messages are clear and actionable, feature works across browsers, no console errors, excellent user experience maintained"
            ]

        for step_idx, step_text in enumerate(steps, start=1):
            fallback_json.append({
                "unique_id": unique_id_start + step_idx,
                "type": "step",
                "step_type": "simple",
                "name": "",
                "teams_udf": "",
                "testcasetype_udf": "",
                "testphase_udf": "",
                "dmnum_udf": "",
                "testjourney_udf": "",
                "description": "",
                "criticality_udf": "",
                "prerequisites_udf": "",
                "testdata_udf": "",
                "circle_udf": "",
                "step_description": step_text
            })

    logger.success(f"Generated {tc_count} IMPROVED fallback test cases with {len(fallback_json)} total rows")
    return fallback_json


def json_to_excel(json_data, jira_id, logger):
    """Convert JSON to Excel"""
    logger.info("Converting JSON to Excel format...")

    rows = []
    for item in json_data:
        row = {}
        for col in EXPECTED_COLS:
            row[col] = item.get(col, "")
        rows.append(row)

    logger.debug(f"Prepared {len(rows)} rows for Excel")

    df = pd.DataFrame(rows, columns=EXPECTED_COLS)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"{(jira_id or 'NOJIRA')}_UAT_TCs_{timestamp}.xlsx"

    filepath = os.path.join(os.getcwd(), filename)

    df.to_excel(filepath, index=False)
    logger.success(f"Excel file created: {filename}")

    return filepath


# -------------------------
# API Call Functions
# -------------------------

def call_bytez_api(api_key, model_name, prompt, logger):
    """Call Bytez API using official Python SDK"""
    logger.progress(f"Calling Bytez API with model: {model_name}...")
    print(f"\n🔷 CALLING BYTEZ API - Model: {model_name}")
    gr.Info(f"🤖 Bytez ({model_name}) is processing...")

    if not BYTEZ_AVAILABLE:
        logger.error("Bytez SDK not installed!")
        gr.Error("❌ Bytez SDK not installed. Run: pip install bytez")
        raise Exception("Bytez SDK not available. Install with: pip install bytez")

    try:
        # Initialize Bytez client
        client = Bytez(api_key=api_key)

        # Get the model
        model = client.model(model_name)

        logger.info(f"Loading model {model_name}...")

        # Prepare the full prompt (system + user message combined)
        full_prompt = (
            "You are a Senior Software Tester. Output ONLY a JSON array. "
            "NO markdown, NO code blocks.\n\n" + prompt
        )

        start_time = time.time()

        # Run the model with parameters
        model_params = {
            "max_new_tokens": 16000,
            "temperature": 0.1,
        }

        logger.debug(f"Sending request to Bytez with {len(full_prompt)} chars")
        result = model.run(full_prompt, model_params=model_params)

        elapsed = time.time() - start_time

        # Check for errors
        if result.error:
            logger.error(f"Bytez API error: {result.error}")
            raise Exception(f"Bytez API error: {result.error}")

        logger.success(f"Bytez completed in {elapsed:.2f}s")
        gr.Info("✅ Bytez response received!")

        # Get the output text
        output_text = result.output

        if isinstance(output_text, list):
            output_text = " ".join(str(item) for item in output_text)

        return output_text.strip()

    except Exception as e:
        logger.error(f"Bytez failed: {str(e)[:200]}")
        gr.Error(f"❌ Bytez Error: {str(e)[:100]}")
        raise


def call_openai_api(api_key, prompt, logger):
    """Call OpenAI API (ChatGPT)"""
    logger.progress("Calling ChatGPT API...")
    print("\n🔷 CALLING CHATGPT API")
    gr.Info("🤖 ChatGPT is processing...")

    try:
        client = OpenAI(api_key=api_key)
        start_time = time.time()

        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a Senior Software Tester. Output ONLY a JSON array. NO markdown, NO code blocks."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=16000,
            timeout=180
        )

        elapsed = time.time() - start_time
        logger.success(f"ChatGPT completed in {elapsed:.2f}s")
        gr.Info("✅ ChatGPT response received!")

        return resp.choices[0].message.content.strip()

    except Exception as e:
        logger.error(f"ChatGPT failed: {str(e)[:200]}")
        gr.Error(f"❌ ChatGPT Error: {str(e)[:100]}")
        raise


def call_gemini_api(api_key, prompt, logger):
    """Call Google Gemini API with safety settings"""
    logger.progress("Calling Gemini API...")
    print("\n🔷 CALLING GEMINI API")
    gr.Info("🤖 Gemini is processing...")

    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel("gemini-pro")

        safety_settings = [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
        ]

        generation_config = {
            "temperature": 0.2,
            "top_p": 0.95,
            "top_k": 40,
            "max_output_tokens": 8192,
        }

        start_time = time.time()

        response = model.generate_content(
            prompt,
            generation_config=generation_config,
            safety_settings=safety_settings
        )

        elapsed = time.time() - start_time
        logger.success(f"Gemini completed in {elapsed:.2f}s")
        gr.Info("✅ Gemini response received!")

        return response.text.strip()

    except Exception as e:
        error_str = str(e)
        logger.error(f"Gemini failed: {error_str[:200]}")

        if "dangerous_content" in error_str.lower():
            logger.error("Gemini blocked response due to safety filters - using fallback")
            gr.Warning("⚠️ Gemini safety filter triggered - using improved fallback mode")
        elif "localhost" in error_str or "Read timed out" in error_str:
            logger.error("Colab localhost timeout - known SDK issue")
            gr.Error("❌ Colab networking issue. Try ChatGPT instead.")
        elif "404" in error_str or "not found" in error_str.lower():
            logger.error("Gemini model not found - trying alternate model names")
            gr.Error("❌ Gemini model issue. Recommendation: Use ChatGPT for best results.")
        else:
            gr.Error(f"❌ Gemini Error: {error_str[:100]}")

        raise


def call_copilot_api(api_key, prompt, logger):
    """Call Microsoft Copilot API"""
    logger.progress("Calling Copilot API...")
    print("\n🔷 CALLING COPILOT API")
    gr.Info("🤖 Copilot is processing...")

    try:
        client = OpenAI(api_key=api_key, base_url="https://api.openai.com/v1")
        start_time = time.time()

        resp = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are a Senior Software Tester. Output ONLY a JSON array."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=16000,
            timeout=180
        )

        elapsed = time.time() - start_time
        logger.success(f"Copilot completed in {elapsed:.2f}s")
        gr.Info("✅ Copilot response received!")

        return resp.choices[0].message.content.strip()

    except Exception as e:
        logger.error(f"Copilot failed: {str(e)[:200]}")
        gr.Error(f"❌ Copilot Error: {str(e)[:100]}")
        raise


def call_openrouter_api(api_key, prompt, logger):
    """Call OpenRouter API"""
    logger.progress("Calling OpenRouter API...")
    print("\n🔷 CALLING OPENROUTER API")
    gr.Info("🤖 OpenRouter is processing...")

    try:
        client = OpenAI(
            api_key=api_key,
            base_url="https://openrouter.ai/api/v1"
        )
        start_time = time.time()

        resp = client.chat.completions.create(
            model="openai/gpt-4o",
            messages=[
                {"role": "system", "content": "You are a Senior Software Tester. Output ONLY a JSON array."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=16000,
            timeout=180
        )

        elapsed = time.time() - start_time
        logger.success(f"OpenRouter completed in {elapsed:.2f}s")
        gr.Info("✅ OpenRouter response received!")

        return resp.choices[0].message.content.strip()

    except Exception as e:
        logger.error(f"OpenRouter failed: {str(e)[:200]}")
        gr.Error(f"❌ OpenRouter Error: {str(e)[:100]}")
        raise


# -------------------------
# Main generation function
# -------------------------

def generate_tcs(
    ai_model, api_key, custom_prompt_toggle, custom_prompt,
    jira_id, design_doc, story_text, txt_file,
    progress=gr.Progress()
):
    """Main test case generation"""

    logger = ConsoleLogger()
    logger.info("========== TEST CASE GENERATION STARTED ==========")
    gr.Info("🚀 Test case generation initiated!")

    progress(0, desc="🔍 Validating inputs...")
    logger.progress("Step 1/6: Validating inputs...")

    # Get model configuration
    model_config = MODEL_CONFIG.get(ai_model)
    if not model_config:
        logger.error(f"Unknown AI model: {ai_model}")
        return (f"❌ Unknown AI model: {ai_model}", None, None, logger.get_html())

    # Check if API key is required
    if model_config["requires_api_key"]:
        if not api_key or len(api_key.strip()) == 0:
            logger.error("API key is required for this model")
            gr.Warning("Please enter an API key.")
            return ("❌ Please enter an API key for this model.", None, None, logger.get_html())
        api_key = api_key.strip()
        logger.success(f"API key provided for {ai_model}")
    else:
        # Use built-in API key for free models
        api_key = model_config["api_key"]
        logger.success(f"Using FREE model: {ai_model} (no API key required)")
        gr.Info(f"🎉 Using FREE model: {ai_model}")

    progress(0.1, desc="📄 Processing input...")
    logger.progress("Step 2/6: Processing user story input...")
    gr.Info("📄 Processing your user story...")

    if txt_file is not None:
        logger.info("Reading story from uploaded file...")
        try:
            if isinstance(txt_file, bytes):
                story = txt_file.decode("utf-8")
            else:
                story = txt_file
            logger.success(f"File read successfully ({len(story)} characters)")
        except Exception as e:
            logger.error(f"File reading failed: {str(e)}")
            return (f"❌ Error reading file: {e}", None, None, logger.get_html())
    else:
        story = story_text or ""
        if story:
            logger.success(f"Using pasted story ({len(story)} characters)")

    if not story.strip():
        logger.error("User story is empty")
        gr.Warning("Please provide a JIRA story.")
        return ("❌ Please provide a JIRA story", None, None, logger.get_html())

    if design_doc is not None:
        doc_count = len(design_doc) if isinstance(design_doc, list) else 1
        logger.info(f"{doc_count} design document(s) uploaded - will be referenced in test cases")
        gr.Info(f"📐 {doc_count} design document(s) detected and will be considered")

    logger.success(f"Story validated ({len(story)} characters)")

    progress(0.2, desc="✍️ Preparing prompt...")
    logger.progress("Step 3/6: Preparing AI prompt...")

    base_prompt = custom_prompt.strip() if custom_prompt_toggle and custom_prompt.strip() else DEFAULT_MASTER_PROMPT
    logger.info("Using CUSTOM prompt" if custom_prompt_toggle else "Using default comprehensive prompt")

    prompt = (
        base_prompt + "\n\n---\n### USER STORY TO ANALYZE:\n\n"
        "JIRA ID: " + (jira_id or "Not provided") + "\n\n" + story
    )

    if design_doc is not None:
        doc_count = len(design_doc) if isinstance(design_doc, list) else 1
        prompt += f"\n\n### DESIGN DOCUMENTS PROVIDED:\nUser has uploaded {doc_count} design mockup(s)/wireframe(s). Consider UI/UX test cases based on design specifications and visual requirements."

    logger.success(f"Prompt prepared ({len(prompt)} chars)")

    progress(0.3, desc=f"🤖 Calling {ai_model}...")
    logger.progress(f"Step 4/6: Calling {ai_model} API...")
    gr.Info(f"🤖 {ai_model} is working on your test cases...")

    try:
        progress(0.4, desc="⏳ Generating (30-120 seconds)...")

        api_type = model_config["api_type"]

        if api_type == "bytez":
            model_text = call_bytez_api(api_key, model_config["model_name"], prompt, logger)
        elif api_type == "openai":
            model_text = call_openai_api(api_key, prompt, logger)
        elif api_type == "gemini":
            model_text = call_gemini_api(api_key, prompt, logger)
        elif api_type == "copilot":
            model_text = call_copilot_api(api_key, prompt, logger)
        elif api_type == "openrouter":
            model_text = call_openrouter_api(api_key, prompt, logger)
        else:
            return (f"❌ Unknown API type: {api_type}", None, None, logger.get_html())

        progress(0.7, desc="📊 Parsing JSON...")
        logger.progress("Step 5/6: Parsing AI response...")
        gr.Info("✅ Response received. Parsing JSON...")

        parsed = extract_json_from_text(model_text, logger)

        if parsed is None:
            logger.error("Failed to parse JSON")
            gr.Error("Failed to parse JSON. Using fallback.")
            raise Exception("JSON parsing failed")

        logger.success("JSON parsed successfully!")

        if isinstance(parsed, dict):
            parsed = [parsed]

        tc_count = len([x for x in parsed if x.get("type") == "test_manual"])
        logger.info(f"Found {tc_count} test cases")
        gr.Info(f"✅ {tc_count} test cases generated.")

        progress(0.8, desc="📋 Creating Excel...")
        logger.progress("Step 6/6: Converting to Excel...")
        gr.Info("📋 Creating Excel file...")

        excel_path = json_to_excel(parsed, jira_id, logger)

        progress(1.0, desc="✅ Complete!")
        logger.success("========== GENERATION COMPLETED SUCCESSFULLY ==========")
        gr.Info(f"🎉 Successfully generated {tc_count} test cases!")

        json_preview = json.dumps(parsed, indent=2)
        if len(json_preview) > 20000:
            json_preview = json_preview[:20000] + "\n\n... (truncated)"

        return (
            f"✅ Generated {tc_count} test cases with {len(parsed)} total rows!\n📥 Excel ready for download.",
            excel_path,
            json_preview,
            logger.get_html()
        )

    except Exception as e:
        error_msg = str(e)
        logger.error(f"Generation failed: {error_msg[:300]}")

        logger.warning("Using IMPROVED fallback mode...")
        gr.Warning("⚠️ API issue - using improved fallback mode...")

        fallback_json = create_fallback_json_from_story(story, jira_id, logger)
        excel_path = json_to_excel(fallback_json, jira_id, logger)

        progress(1.0, desc="✅ Fallback complete")
        logger.warning("========== COMPLETED WITH FALLBACK MODE ==========")
        gr.Info("✅ Fallback generation complete.")

        tc_count = len([x for x in fallback_json if x.get("type") == "test_manual"])
        json_preview = json.dumps(fallback_json, indent=2)[:20000]

        status_msg = f"⚠️ API Error: {error_msg[:150]}\n\n✅ Generated {tc_count} IMPROVED template test cases as fallback."

        return (status_msg, excel_path, json_preview, logger.get_html())


# -------------------------
# Model selection handler
# -------------------------
def update_api_key_visibility(selected_model):
    """Show/hide API key field based on selected model"""
    model_config = MODEL_CONFIG.get(selected_model, {})
    requires_key = model_config.get("requires_api_key", True)

    if requires_key:
        return gr.update(visible=True, label="API Key (Required)", placeholder="Enter your API key here...")
    else:
        return gr.update(visible=False, label="API Key (Not Required - Using Free Model)", placeholder="No API key needed for this free model")


# -------------------------
# Clear all function
# -------------------------
def clear_all():
    return (
        "Bytez GPT-4o-mini (FREE)", "", False, "", "", [], "", None, "", None, "",
        "<div style='position: relative;'><div class='console-box' id='console-logs-content'>✨ Cleared. Ready for new generation.</div></div>"
    )


# -------------------------
# Build Gradio UI
# -------------------------

with gr.Blocks(title="🚀 UAT Test Case Generator") as demo:

    # Inject custom CSS styling
    gr.HTML("""
    <style>
    body {
        background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
        font-family: 'Segoe UI', Roboto, Arial, sans-serif;
    }
    .console-box {
        background: #1e1e1e;
        border-radius: 8px;
        padding: 16px;
        padding-top: 40px;
        font-family: 'Courier New', monospace;
        font-size: 13px;
        line-height: 1.6;
        max-height: 500px;
        overflow-y: auto;
        border: 1px solid #333;
    }
    .gr-button {
        border-radius: 12px !important;
        font-weight: 600 !important;
        flex: 1 1 0 !important;
        min-width: 0 !important;
    }
    #proceed-btn {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
        color: white !important;
    }
    #clear-btn {
        background: linear-gradient(135deg, #f59e0b 0%, #d97706 100%) !important;
        color: white !important;
    }
    #refresh-btn {
        background: linear-gradient(135deg, #10b981 0%, #059669 100%) !important;
        color: white !important;
    }
    .important-note {
        background: #fef3c7;
        border-left: 4px solid #f59e0b;
        padding: 15px;
        border-radius: 8px;
        margin: 15px 0;
    }
    .api-guide {
        background: linear-gradient(135deg, #e0f2fe 0%, #dbeafe 100%);
        border-left: 4px solid #3b82f6;
        padding: 20px;
        border-radius: 8px;
        margin: 20px 0;
    }
    .api-setup-accordion {
        background: linear-gradient(135deg, #fef3c7 0%, #fde68a 100%) !important;
        border: 3px solid #f59e0b !important;
        border-radius: 8px !important;
        padding: 4px !important;
        box-shadow: 0 4px 6px rgba(245, 158, 11, 0.3) !important;
        animation: pulse-border 2s ease-in-out infinite;
    }
    @keyframes pulse-border {
        0%, 100% { box-shadow: 0 4px 6px rgba(245, 158, 11, 0.3); }
        50% { box-shadow: 0 4px 12px rgba(245, 158, 11, 0.6); }
    }
    .api-setup-accordion summary {
        background: linear-gradient(135deg, #fbbf24 0%, #f59e0b 100%) !important;
        color: #7c2d12 !important;
        font-weight: 700 !important;
        font-size: 16px !important;
        padding: 12px 16px !important;
        border-radius: 6px !important;
    }
    .api-setup-accordion summary:hover {
        background: linear-gradient(135deg, #f59e0b 0%, #d97706 100%) !important;
        color: white !important;
    }
    .api-card {
        background: white;
        padding: 15px;
        margin: 10px 0;
        border-radius: 8px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.1);
    }
    .copy-btn {
        background: #3b82f6;
        color: white;
        border: none;
        padding: 4px 8px;
        border-radius: 4px;
        cursor: pointer;
        font-size: 11px;
        margin-left: 8px;
    }
    .creator-credit {
        text-align: center;
        padding: 15px;
        margin-top: 20px;
        background: linear-gradient(135deg, #f3f4f6 0%, #e5e7eb 100%);
        border-radius: 8px;
        border: 2px solid #d1d5db;
    }
    .cta-help-text {
        font-size: 12px;
        color: #6b7280;
        text-align: center;
        margin-top: 8px;
        font-style: italic;
    }
    </style>
    """)

    gr.Markdown("""
    # 🚀 UAT Test Case Generator
    ### Professional Test Case Generation powered by Advanced AI Models
    """)

    gr.HTML("""
    <div class="important-note">
        <strong>⚠️ SETUP REQUIRED:</strong><br>
        • <strong>Install Bytez SDK first:</strong> Run <code>pip install bytez</code> in your terminal<br>
        • <strong>Then use FREE models:</strong> Bytez GPT-4o-mini and Gemma-3-1b - No additional API key needed!<br>
        • <strong>For maximum test cases (20-35 TCs):</strong> Use <strong>ChatGPT</strong> (requires your own API key)<br>
        • Check <strong>console logs</strong> below for detailed real-time progress
    </div>
    """)

    gr.HTML("""
    <div style="background: linear-gradient(135deg, #dcfce7 0%, #bbf7d0 100%);
                border: 3px solid #16a34a;
                border-radius: 8px;
                padding: 15px;
                margin: 15px 0;
                box-shadow: 0 4px 12px rgba(22, 163, 74, 0.4);">
        <div style="text-align: center;">
            <h3 style="color: #14532d; margin: 0; font-size: 18px;">
                🎉 <strong>FREE MODELS AVAILABLE!</strong> 🎉
            </h3>
            <p style="color: #166534; margin: 8px 0 0 0; font-size: 14px;">
                ✨ <strong>Bytez GPT-4o-mini (FREE)</strong> - Best quality free model<br>
                ✨ <strong>Bytez Gemma-3-1b (FREE)</strong> - Lightning fast<br>
                <br>
                <strong style="color: #dc2626;">⚠️ First run: pip install bytez</strong>
            </p>
        </div>
    </div>
    """)

    with gr.Accordion("🔑 How to Get API Keys (Only for Paid Models)", open=False, elem_classes="api-setup-accordion"):
        gr.HTML("""
        <div class="api-guide">
            <div style="background: #dcfce7; padding: 15px; border-radius: 8px; margin-bottom: 20px; border-left: 4px solid #16a34a;">
                <h3 style="color: #14532d; margin: 0;">✅ FREE Models (No API Key Required)</h3>
                <p style="color: #166534; margin: 10px 0;">
                    <strong>Setup:</strong> Run <code>pip install bytez</code> first<br>
                    <strong>🎉 Bytez GPT-4o-mini (FREE)</strong> - Premium quality, no signup<br>
                    <strong>🎉 Bytez Gemma-3-1b (FREE)</strong> - Fast responses, completely free
                </p>
            </div>

            <h3>API Key Setup for Paid Models</h3>
            <p>For ChatGPT, Gemini, and other paid models, refer to their respective documentation for API key setup.</p>
        </div>
        """)

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("### 🤖 AI Model Configuration")

            ai_model = gr.Dropdown(
                choices=list(MODEL_CONFIG.keys()),
                value="Bytez GPT-4o-mini (FREE)",
                label="AI Model (FREE models marked)"
            )

            api_key = gr.Textbox(
                value="",
                placeholder="No API key needed for free models",
                type="password",
                label="API Key (Not Required - Using Free Model)",
                visible=False
            )

            # Add change event for model selection
            ai_model.change(
                fn=update_api_key_visibility,
                inputs=[ai_model],
                outputs=[api_key]
            )

            gr.Markdown("### 🎯 Custom Prompt (Optional)")
            custom_prompt_toggle = gr.Checkbox(
                label="Use Custom Prompt (Default generates 20-35 comprehensive test cases)",
                value=False
            )
            custom_prompt = gr.Textbox(
                label="Custom Prompt",
                lines=6,
                visible=False
            )

            custom_prompt_toggle.change(
                fn=lambda x: gr.update(visible=x),
                inputs=[custom_prompt_toggle],
                outputs=[custom_prompt]
            )

            gr.Markdown("### 📝 JIRA Information")

            jira_field = gr.Textbox(
                label="JIRA ID (Optional)",
                placeholder="e.g., LRECOM-16429",
                lines=1
            )

            design_doc = gr.File(
                label="📐 Design Documents (Optional - Upload up to 4 mockups/wireframes) | Accepted: .jpg, .jpeg, .png only",
                file_count="multiple",
                type="filepath",
                file_types=["image"]
            )

            gr.Markdown("### 📄 User Story Input")

            with gr.Tab("📋 Paste Story"):
                story_field = gr.Textbox(
                    label="User Story",
                    placeholder="Paste your complete JIRA story here...",
                    lines=15
                )

            with gr.Tab("📤 Upload File"):
                txt_file = gr.File(
                    label="Upload Story Document | Accepted: .pdf, .doc, .docx, .txt only",
                    file_count="single",
                    type="binary",
                    file_types=[".pdf", ".doc", ".docx", ".txt"]
                )

            with gr.Row():
                proceed_btn = gr.Button(
                    "🚀 Proceed",
                    variant="primary",
                    size="lg",
                    elem_id="proceed-btn"
                )
                clear_btn = gr.Button(
                    "🔁 Clear",
                    variant="secondary",
                    size="lg",
                    elem_id="clear-btn"
                )
                refresh_btn = gr.Button(
                    "🔄 Refresh",
                    variant="secondary",
                    size="lg",
                    elem_id="refresh-btn"
                )

            gr.HTML("""
            <div class="cta-help-text">
                <strong>Proceed:</strong> Generate test cases | <strong>Clear:</strong> Reset all fields | <strong>Refresh:</strong> Reload the page
            </div>
            """)

        with gr.Column(scale=2):
            gr.Markdown("### 🖥️ Real-Time Console Logs")
            console_output = gr.HTML(
                value="<div style='position: relative;'><div class='console-box'>⏳ Waiting for generation to start...</div></div>",
                label=""
            )

            status_output = gr.Textbox(
                label="📊 Status Summary",
                interactive=False,
                lines=8
            )

            download_link = gr.File(
                label="📥 Download Excel File",
                interactive=False
            )

            with gr.Accordion("🔍 Raw JSON Preview", open=False):
                raw_json_out = gr.Code(
                    label="",
                    language="json",
                    lines=12,
                    interactive=False
                )

    gr.HTML("""
    <div style='text-align: center; color: #6b7280; font-size: 13px; padding: 15px 20px; margin-top: 20px;'>
        <div style='margin-bottom: 10px;'>
            🔍 <b>Console Colors:</b> <span style='color: #16a34a;'>Green = Success</span> |
            <span style='color: #dc2626;'>Red = Error</span> |
            <span style='color: #ea580c;'>Orange = Warning</span> |
            <span style='color: #2563eb;'>Blue = Info</span> |
            <span style='color: #06b6d4;'>Cyan = Progress</span>
        </div>
    </div>
    <div class='creator-credit'>
        <strong>Created by <a href='https://www.linkedin.com/in/sunilbarigala/' target='_blank'>Sunil Barigala</a> using Vibe Coding AI Models 🚀</strong><br>
        <span style="color: #16a34a; font-size: 12px;">✨ Enhanced with Bytez FREE Models</span>
    </div>
    """)

    proceed_btn.click(
        fn=generate_tcs,
        inputs=[
            ai_model, api_key, custom_prompt_toggle, custom_prompt,
            jira_field, design_doc, story_field, txt_file
        ],
        outputs=[status_output, download_link, raw_json_out, console_output]
    )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[
            ai_model, api_key, custom_prompt_toggle, custom_prompt,
            jira_field, design_doc, story_field, txt_file,
            status_output, download_link, raw_json_out, console_output
        ]
    )

    refresh_btn.click(
        fn=None,
        inputs=[],
        outputs=[],
        js="() => { window.location.reload(); }"
    )

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🚀 UAT TEST CASE GENERATOR")
    print("="*80)
    print("⚠️ FIRST TIME SETUP: pip install bytez")
    print("🎉 FREE MODELS AVAILABLE - NO API KEY NEEDED!")
    print("✨ Bytez GPT-4o-mini (FREE) - Premium quality")
    print("✨ Bytez Gemma-3-1b (FREE) - Lightning fast")
    print("📍 Optimized for Google Colab")
    print("🔍 Check console for detailed logs")
    print("="*80 + "\n")

    demo.launch(
        share=True,
        debug=True,
        inline=False
    )

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


⚠️ WARNING: Bytez SDK not installed. Install with: pip install bytez

🚀 UAT TEST CASE GENERATOR
⚠️ FIRST TIME SETUP: pip install bytez
🎉 FREE MODELS AVAILABLE - NO API KEY NEEDED!
✨ Bytez GPT-4o-mini (FREE) - Premium quality
✨ Bytez Gemma-3-1b (FREE) - Lightning fast
📍 Optimized for Google Colab
🔍 Check console for detailed logs

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dc36903d9de5cc41aa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
ℹ️ INFO: ========== TEST CASE GENERATION STARTED ==========
🔄 PROGRESS: Step 1/6: Validating inputs...
❌ ERROR: API key is required for this model
